In [1]:
import findspark
findspark.init()

from pyspark.conf import SparkConf
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

conf = SparkConf().setAppName("1174").setMaster("local[4]")
spark = SparkSession.builder.config(conf=conf).getOrCreate()
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/08/06 01:47:02 WARN Utils: Your hostname, de24, resolves to a loopback address: 127.0.1.1; using 192.168.0.102 instead (on interface enp0s3)
25/08/06 01:47:02 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/08/06 01:47:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/08/06 01:47:07 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [ ]:
'''
Table: Delivery

+-----------------------------+---------+
| Column Name                 | Type    |
+-----------------------------+---------+
| delivery_id                 | int     |
| customer_id                 | int     |
| order_date                  | date    |
| customer_pref_delivery_date | date    |
+-----------------------------+---------+
delivery_id is the column of unique values of this table.
The table holds information about food delivery to customers that make orders at some date and specify 
a preferred delivery 
date (on the same order date or after it).
 

If the customer's preferred delivery date is the same as the order date, then the order is called 
immediate; otherwise, 
it is called scheduled.

The first order of a customer is the order with the earliest order date that the customer made. 
It is guaranteed that a customer has precisely one first order.

Write a solution to find the percentage of immediate orders in the first orders of all customers, 
rounded to 2 decimal places.

The result format is in the following example.

 

Example 1:

Input: 
Delivery table:
+-------------+-------------+------------+-----------------------------+
| delivery_id | customer_id | order_date | customer_pref_delivery_date |
+-------------+-------------+------------+-----------------------------+
| 1           | 1           | 2019-08-01 | 2019-08-02                  |
| 2           | 2           | 2019-08-02 | 2019-08-02                  |
| 3           | 1           | 2019-08-11 | 2019-08-12                  |
| 4           | 3           | 2019-08-24 | 2019-08-24                  |
| 5           | 3           | 2019-08-21 | 2019-08-22                  |
| 6           | 2           | 2019-08-11 | 2019-08-13                  |
| 7           | 4           | 2019-08-09 | 2019-08-09                  |
+-------------+-------------+------------+-----------------------------+
Output: 
+----------------------+
| immediate_percentage |
+----------------------+
| 50.00                |
+----------------------+
Explanation: 
The customer id 1 has a first order with delivery id 1 and it is scheduled.
The customer id 2 has a first order with delivery id 2 and it is immediate.
The customer id 3 has a first order with delivery id 5 and it is scheduled.
The customer id 4 has a first order with delivery id 7 and it is immediate.
Hence, half the customers have immediate first orders.
'''

In [2]:
data = [
(1,1,'2019-08-01','2019-08-02'),
(2,2,'2019-08-02','2019-08-02'),
(3,1,'2019-08-11','2019-08-12'),
(4,3,'2019-08-24','2019-08-24'),
(5,3,'2019-08-21','2019-08-22'),
(6,2,'2019-08-11','2019-08-13'),
(7,4,'2019-08-09','2019-08-09')    
]
schema = ["delivery_id","customer_id","order_date","customer_pref_delivery_date"]

In [3]:
df = spark.createDataFrame(data = data, schema = schema)
df.show()

+-----------+-----------+----------+---------------------------+
|delivery_id|customer_id|order_date|customer_pref_delivery_date|
+-----------+-----------+----------+---------------------------+
|          1|          1|2019-08-01|                 2019-08-02|
|          2|          2|2019-08-02|                 2019-08-02|
|          3|          1|2019-08-11|                 2019-08-12|
|          4|          3|2019-08-24|                 2019-08-24|
|          5|          3|2019-08-21|                 2019-08-22|
|          6|          2|2019-08-11|                 2019-08-13|
|          7|          4|2019-08-09|                 2019-08-09|
+-----------+-----------+----------+---------------------------+



In [4]:
from pyspark.sql.window import Window

In [22]:
window_spec = Window.partitionBy(F.col("customer_id")).orderBy(F.col("order_date"))

df.withColumn("rnk", F.rank().over(window_spec))\
  .where(F.col("rnk") == 1)\
  .select( F.round(
              (F.sum( F.when(F.col("order_date") == F.col("customer_pref_delivery_date"), 1).otherwise(0)) / F.count(F.col('*')))*100,
             2).alias("immediate_percentage") )\
  .show()

+--------------------+
|immediate_percentage|
+--------------------+
|                50.0|
+--------------------+



## SQL Solution
<pre>
WITH FIRST_ORDER AS (
    SELECT customer_id,order_date,customer_pref_delivery_date,
           RANK() OVER(PARTITION BY customer_id order by order_date ) as rnk
    FROM Delivery
)
SELECT ROUND((SUM( CASE WHEN customer_pref_delivery_date = order_date THEN 1 ELSE 0 END)/count(*))*100,2) as immediate_percentage 
FROM FIRST_ORDER 
WHERE rnk =1
</pre>